**In this tutorial, we learn how to use GPUs to facilitate large-scale analyses of multiple annual reports.**

https://share.google/96W3gl4njTYizHovl

do i need to set runtime to GPU vscode on macbook air M2 chip
no but...
However, if you are doing machine learning (PyTorch/TensorFlow) or rendering, you need to configure your Python environment to use MPS (Metal Performance Shaders), Apple's equivalent to CUDA for GPU acceleration. 

In [1]:
# standard libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline # for using the models

import spacy # for sentence extraction
from tika import parser # for the report extraction

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tika/__init__.py:20: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__('pkg_resources').declare_namespace(__name__)


In [2]:
### Load the models (takes ca. 1 min)
# Environmental model.
name = "ESGBERT/EnvironmentalBERT-environmental" # path to download from HuggingFace
# In simple words, the tokenizer prepares the text for the model and the model classifies the text-
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)
# The pipeline combines tokenizer and model to one process.
pipe_env = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0) # set device=0 to use GPU

# Action model.
name = "ESGBERT/EnvironmentalBERT-action"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSequenceClassification.from_pretrained(name)
pipe_act = pipeline("text-classification", model=model, tokenizer=tokenizer, device=0) # set device=0 to use GPU

### IMPORTANT: SET RUNTIME TO GPU (see above)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/805 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/328M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [3]:
# Tryout the model
pipe_env("We are really relying on people improving their consumer decisions to fight climate change.")

[{'label': 'environmental', 'score': 0.9977815747261047}]

In [4]:
pipe_act("We planted 7.500 trees in the last 5 years.")

[{'label': 'action', 'score': 0.9999480247497559}]

## Step 2: Set up PDF extraction pipeline

We use the code from [Tutorial 1](https://medium.com/p/8daa2695f6c5) to set up a pipeline to transform PDF to texts.

In [5]:
# Encapsulate code from tutorial 1 in a function.
def PDFtoSentence(path):
  print(f"\nParsing {path}")
  # The from_file() function of tika helps us to load the content of the document. (take ca. 30 sec)
  print("- PDF to txt")
  report = parser.from_file(path)
  # For this, we use the nlp() function from spacy. (takes 20 secs)
  print("- txt to sentences")
  nlp = spacy.load('en_core_web_sm')
  about_doc = nlp(report["content"][:1000000])
  # One downside of spacy is that it can only parse 1.000.000 signs at a time.
  # You can write a for-loop around the report["content"] or use other tools.
  # For simplicitiy, we only use the first 1.000.000 characters.

  # We transfer the sequences ("about_doc.sents") to a list of raw strings.
  sequences = list(map(str, about_doc.sents))
  # "\n" signals a new line. We remove this so that the output looks better.
  sentences = [x.replace("\n", "") for x in sequences]
  # Remove all empty text, i.e. if the value is "", i.e are empty.
  sentences = [x for x in sentences if x != ""]
  # A sentence should start with upper case.
  sentences = [x for x in sentences if x[0].isupper()]
  return sentences

In [6]:
# Example reports
ogfi25 = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/Green Finance Impact Report 2025.pdf'
jpmc24 = '/Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/jpmc-sustainability-report-2024.pdf'
reports = [ogfi25, jpmc24]

In [7]:
# Run PDF to sentence.
ogfi25_sent = PDFtoSentence(ogfi25)
ogfi25_sent


Parsing /Users/elisabethwatson/Birkbeck/PGC ADS/Module 3 Project/Datasets/GBTP 260201/Green Finance Impact Report 2025.pdf
- PDF to txt
- txt to sentences


['Green Finance Impact  Report 2025ContentHighlights 2025\t 3CFO foreword\t 4Our green bonds\t 5–  Projects with green bond allocations\t 6–  Total amounts allocated by bond\t 7–  Green bond allocations by project – 2017-2025\t 8Our green loans\t 10Sustainability impact from our green proceeds\t 11–  Climate impact\t 12–  Biodiversity impact\t 13–  Community impact\t 14 Statement by the Executive Board\t 15Independent limited assurance report  on Selected Information in the Green Finance Impact Report\t 16Appendix I: Accounting policies\t 18Summarised blue bond impacts 2024',
 'This document is a public summary of Ørsted’s  Blue Bond Impact Report 2024.',
 'The full report,  including information on the allocation of net  proceeds, is available solely to blue bond investors.',
 'Summarised Blue Bond Impacts 2024→Other relevant  publicationsBiodiversity  Measurement Framework→ØrstedWe have launched a next-generation framework for holistically  measuring and reporting biodiversity impact